> 📒 이 노트북은 [펭귄 데이터로 배우는 인공지능](https://yanghyeokjin33.github.io/Algorithm/) 수업에서 쓰는 실습 노트북이에요. 원본: [https://github.com/YANGHYEOKJIN33/Algorithm](https://github.com/YANGHYEOKJIN33/Algorithm)

# 🧩 데이터 통합과 훈련/테스트 분할

**목표**: 여러 표를 하나로 합치고(concat·merge), 합친 표를 입력 X와 정답 y, 훈련 데이터와 테스트 데이터로 나눈다.

> 🔁 **만약에 — 데이터가 여러 파일로 온다면?** 1단원에서는 7쪽을 한 표로 모았지만, 실제로는 표가 여러 파일로 오는 일이 많아요. 1·2는 그 연습이고, 2에서 합친 깨끗한 341줄을 3에서 나눠요.

> **Colab 사용법** — 셀 왼쪽의 ▶ 단추(또는 **Shift + Enter**)를 누르면 그 셀이 실행돼요.
> 위에서부터 **차례로** 실행하세요. 고쳐 보고 싶으면 먼저 **파일 → 드라이브에 사본 저장**을 눌러 내 것으로 만드세요.
> 파이썬을 몰라도 괜찮아요. 코드 위의 설명과 사이트의 의사코드를 짝지어 읽으면 돼요.

- 판다스만 있으면 돼요.

In [ ]:
import pandas as pd
SITE = 'https://yanghyeokjin33.github.io/Algorithm/practice'

## 1. 세로로 이어 붙이기 — `pd.concat`

`pd.read_html(주소)`는 웹 페이지의 <table>을 바로 데이터프레임으로 읽어 주는 지름길이에요(크롤링 한 줄 요약!). 쪽마다 따로 받은 표 7개를 위아래로 이어 붙여요.

- read_html은 표 목록을 돌려줘서 [0]으로 첫 번째 표를 꺼내요.
- ignore_index=True를 주면 이어 붙인 뒤 인덱스를 0, 1, 2, …로 새로 매겨요.
- 이 345줄은 1단원에서 모은 표와 같아요. 3단원에서 다듬어 깨끗한 341줄이 됐어요.

In [ ]:
pages = []
for p in range(1, 8):
    t = pd.read_html(f'{SITE}/page{p}.html', encoding='utf-8')[0]   # 그 쪽의 첫 번째 표
    pages.append(t)
print([len(t) for t in pages])

crawl = pd.concat(pages)
print(crawl.index[:3], crawl.index[50:53])     # 인덱스가 0, 1, 2 … 다시 0, 1, 2 … 겹쳐요
crawl = pd.concat(pages, ignore_index=True)    # 인덱스를 0부터 새로
print(crawl.shape)

## 2. 열쇠로 옆에 붙이기 — `pd.merge`

🔁 **만약에** 측정값과 종이 다른 파일로 왔다면? 3단원의 깨끗한 341줄을 연습용으로 두 파일에 나눠 두었어요. 측정표(번호·부리·날개·몸무게 — 종 칸이 없어요)와 판정표(번호·종)를 **번호**가 같은 행끼리 짝지어 옆으로 붙여요.

- 측정표에는 종 칸이 없고, 판정표에는 번호와 종만 있어요.
- 판정표는 순서가 뒤섞여 있어요. 그래도 번호(열쇠)만 같으면 같은 펭귄이에요.

In [ ]:
measure = pd.read_csv('https://raw.githubusercontent.com/YANGHYEOKJIN33/Algorithm/main/data/penguins_measure.csv')
label = pd.read_csv('https://raw.githubusercontent.com/YANGHYEOKJIN33/Algorithm/main/data/penguins_label.csv')
print('측정표', measure.shape, '/ 판정표', label.shape)
label.head()

- on='번호'가 열쇠예요. 순서가 달라도 번호로 짝을 찾아요.
- how='left'는 왼쪽 표의 행을 모두 남기고, 짝이 없으면 NaN으로 채워요.
- 341마리가 모두 짝을 찾아서 두 방법 모두 341줄이에요. (짝이 없는 경우는 사이트 4-2의 100번 펭귄에서 봤어요.)

In [ ]:
df = pd.merge(measure, label, on='번호')               # 기본: 두 표에 다 있는 번호만(inner)
print('합친 표:', df.shape)
print('짝을 못 찾아 빠진 펭귄:', len(measure) - len(df))
left = pd.merge(measure, label, on='번호', how='left') # 측정표는 모두 남기기
print('how=left:', left.shape, '→ 종이 빈 행', left['종'].isnull().sum())
df.head()

## 3. 훈련 데이터와 테스트 데이터로 나누기

2에서 번호로 합친 표 `df`(341줄)를 그대로 나눠요. 입력 **X**(대문자, 여러 열 — 3-1에서 고른 속성 4개)와 정답 **y**(소문자, 한 열)로 나눈 뒤, 섞어서 80% : 20%로 나눠요.

- test_size=0.2는 20%를 테스트로 떼어 두라는 뜻이에요.
- random_state=42는 섞는 방법을 고정해 언제 실행해도 같은 결과가 나오게 해요(사이트의 "씨앗 고정").
- X와 y는 같은 순서로 섞여서 짝이 깨지지 않아요.
- 합친 표는 깨끗한 표(penguins_clean.csv)와 줄 순서까지 같아서, 5단원 노트북이 깨끗한 표를 불러와 똑같이 나눠도 같은 훈련 272줄 · 테스트 69줄이 나와요.

In [ ]:
from sklearn.model_selection import train_test_split

X = df[['부리길이', '부리깊이', '날개길이', '몸무게']]
y = df['종']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
print('훈련:', X_train.shape, ' 테스트:', X_test.shape)
print(y_train.value_counts())

## 🧩 확인해 봐요

- test_size를 0.3으로 바꾸면 테스트 데이터는 몇 행이 될까요?
- `stratify=y`를 더하면 훈련·테스트의 종 비율이 원래와 같아져요. 더해 보고 `y_test.value_counts()`를 견줘 보세요.